# SI26 — Week 6: Code Switching Dataset Collection
### Code Saviours Summer Internship 2026 — Project 2 Kickoff
**Author:** Qandeel Asim

**Goal:** Build a Roman Urdu–English code-switched sentence dataset (150+ sentences, word-level labels: `URD` / `ENG` / `MIX`), publish it on GitHub + Hugging Face.

**Approach used in this notebook:** instead of scraping Twitter/Reddit live (which now needs paid API access / logins), we pull real, already-collected Roman Urdu sentences from a public research dataset (Sharf, 2017 — UCI ML Repository, CC BY 4.0), then filter that down to the sentences that are genuinely code-switched (mix Urdu + English), and auto-label each word. No signup, no API key — everything downloads directly inside Colab.


## Step 1 — GitHub Repo + Colab Notebook (manual, do this outside Colab)

1. Create a new GitHub repo named: `code-switching-codesaviours-si26-qandeel`
2. Add a README with: `Code Switching NLP | Code Saviours SI-26 | Qandeel Asim`
3. Save/rename this notebook as: `SI26-Week6-Qandeel.ipynb` (hyphens, not underscores) and push it to the repo.


In [1]:
import pandas as pd
import re
import random

## Step 2 — Collect Real Code-Switched Sentences (automatic, no login)

**Source:** [Roman Urdu Data Set](https://archive.ics.uci.edu/dataset/458/roman+urdu+data+set) (Sharf, 2017), mirrored on GitHub — 20,000+ real sentences gathered from e-commerce reviews, public Facebook page comments, and Twitter posts. Licensed CC BY 4.0 (free to use with attribution).

This is real, already-collected social-media text — not synthetic. We download it directly, then keep only the sentences that actually mix Urdu and English (a lot of the corpus is pure Urdu).

In [2]:
# Download the real dataset directly — no login required
url = "https://raw.githubusercontent.com/Smat26/Roman-Urdu-Dataset/master/Dataset/Roman%20Urdu%20DataSet.csv"
df = pd.read_csv(url, header=None, names=['sentence', 'sentiment', 'extra'],
                  on_bad_lines='skip', engine='python')

df = df.dropna(subset=['sentence'])
df['sentence'] = df['sentence'].astype(str).str.strip()
df = df[df.sentence.str.len() > 0].drop_duplicates(subset='sentence')
print(f"Total real sentences downloaded: {len(df)}")

Total real sentences downloaded: 19588


In [3]:
# Filter for genuinely code-switched sentences (Urdu + English mixed)
urdu_markers = {'hai','hain','ka','ki','ke','tha','thi','nahi','nhi','mera','meri','bhai','yaar',
    'bohot','bahut','kya','kar','raha','rahi','hoon','hun','mein','main','se','ko','par','aur',
    'lekin','sab','bhi','kal','aaj','ye','wo','koi','kuch','acha','achi','bura','buri','tu','tum',
    'aap','hum','unko','uska','uski','iska','iski','sy','nai','hy','ni','wala','wali','bht'}

english_words = {'the','is','was','and','but','so','right','wrong','love','best','nice','good',
 'bad','sad','happy','sorry','please','thanks','thank','you','feel','feeling','time','life','world',
 'true','false','real','fake','proud','miss','missing','care','support','respect','trust','hope',
 'work','job','busy','tired','excited','stressed','boring','amazing','awesome','great','perfect',
 'beautiful','free','vote','election','change','future','past','present','history','story','song',
 'music','movie','actor','actress','player','team','match','game','sport','fan','fans','follow',
 'like','comment','share','post','video','photo','online','internet','phone','mobile','message',
 'text','call','email','plan','order','check','update','problem','issue','result','success','still'}

mixwords = {'phone','internet','mobile','message','order','plan','time','net','system','job'}

def is_code_switched(text):
    words = re.findall(r"[A-Za-z']+", text.lower())
    if not (5 <= len(words) <= 25):
        return False
    has_urdu = sum(1 for w in words if w in urdu_markers) >= 2
    has_eng  = sum(1 for w in words if w in english_words) >= 1
    return has_urdu and has_eng

filtered = df[df.sentence.apply(is_code_switched)].sentence.tolist()
print(f"Real code-switched candidates found: {len(filtered)}")

random.seed(42)
random.shuffle(filtered)
sample = filtered[:200]   # increase/decrease as needed
print(f"Sampled: {len(sample)} sentences")

Real code-switched candidates found: 1773
Sampled: 200 sentences


In [4]:
# Preview a sample before labelling
for s in sample[:15]:
    print(s)

Jin janab ne phone uthaya tha un k mon se E call karne ka shukria・k ilawa kuch nikal hi na paya.
overall phone is good laikan in market bsck cover aor glass nhi mil raha , ye saath bhejhna chahiye tha
Wo guzishta dinon Russia mein hone wale World University moqablon mein Pakistan Universities ki numaindagi kar chuki hain
Mutee-Ur-Rehman name is bangali officer ne tayyarey ka control sambhal liya aur Karachi mein apne 2 sathiyon ko peygham dete hue kaha
May nay 1 buy is price per zaberdast half selve sweater hai high quality hai
Un ki tehrir parh kar yun mehsos hota hai ka is ke alfaz moye qalam se nahi balke dil se nikle hon
BA karne ke bad Molvi Abdul Haq Sir Syed Ahmed Khan ki hidayat par 1895 mein Hyderabad Dakan pohanche
1975 mein inho ne Mohammad Ali aur Chuck Wepner ke darmiyan boxing match dekha
Asif Iqbal ne apna akhri match Eden gardens kolkata mein khela aur is match ki dusri inning mein 15 runs bana kar run out huwe
O bhaijan abhi inka level world ke biggest rock bands se nh

## Step 3 — Word-Level Labelling

**Labelling scheme:**
- `URD` — Urdu word written in Roman script
- `ENG` — English word
- `MIX` — a widely-nativized loanword that functions as Urdu vocabulary despite English origin (e.g. `phone`, `internet`, `mobile`)

The labels below are auto-assigned from word lists, so they're a strong starting point — **not perfect**. Real, informal social-media text has slang and inconsistent spelling, so scroll through `df_final.head(40)` after this and fix any labels that look wrong before submitting.

In [5]:
def label_word(w):
    wl = re.sub(r'[^a-zA-Z]', '', w).lower()
    if not wl:
        return 'URD'
    if wl in mixwords:
        return 'MIX'
    if wl in english_words:
        return 'ENG'
    return 'URD'   # default: corpus is majority Roman Urdu

data = []
for s in sample:
    words = s.split()
    labels = [label_word(w) for w in words]
    data.append({'sentence': s, 'words': words, 'labels': labels})

print(f"Total labelled sentences: {len(data)}")

Total labelled sentences: 200


## Convert to flat CSV format (as required by the assignment)

In [6]:
rows = []
for entry in data:
    for word, label in zip(entry['words'], entry['labels']):
        rows.append({
            'sentence': entry['sentence'],
            'word': word,
            'label': label
        })

df_final = pd.DataFrame(rows)
df_final.to_csv('dataset.csv', index=False, encoding='utf-8')

print(f'Dataset created: {len(df_final)} word entries')
print(f'Sentences: {df_final.sentence.nunique()}')
print('Label distribution:')
print(df_final.label.value_counts())

Dataset created: 3182 word entries
Sentences: 200
Label distribution:
label
URD    2936
ENG     234
MIX      12
Name: count, dtype: int64


In [7]:
df_final.head(40)

,sentence,word,label
0,Jin janab ne phone uthaya tha un k mon se E ca...,Jin,URD
1,Jin janab ne phone uthaya tha un k mon se E ca...,janab,URD
2,Jin janab ne phone uthaya tha un k mon se E ca...,ne,URD
3,Jin janab ne phone uthaya tha un k mon se E ca...,phone,MIX
4,Jin janab ne phone uthaya tha un k mon se E ca...,uthaya,URD
5,Jin janab ne phone uthaya tha un k mon se E ca...,tha,URD
6,Jin janab ne phone uthaya tha un k mon se E ca...,un,URD
7,Jin janab ne phone uthaya tha un k mon se E ca...,k,URD
8,Jin janab ne phone uthaya tha un k mon se E ca...,mon,URD
9,Jin janab ne phone uthaya tha un k mon se E ca...,se,URD


## Step 4 — Publish Dataset on Hugging Face (manual, do this outside Colab)

1. Go to https://huggingface.co/new-dataset
2. Dataset name: `code-switching-codesaviours-si26-qandeel`
3. Visibility: **Public**
4. Upload the `dataset.csv` generated above
5. Write a dataset card covering:
   - **What it is** — a word-level labelled Roman Urdu–English code-switching dataset
   - **How it was collected** — real sentences sourced from the Roman Urdu Data Set (Sharf, 2017, UCI ML Repository, CC BY 4.0 — originally gathered from e-commerce reviews, Facebook comments, and Twitter posts), filtered down to code-switched sentences and word-labelled
   - **Label meanings** — `URD` = Roman Urdu word, `ENG` = English word, `MIX` = nativized English loanword used as Urdu vocabulary
   - **Attribution** — credit: Sharf, Z. (2017). *Roman Urdu Data Set* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C58325


## Submission Checklist (Friday deadline)

- [ ] GitHub repo `code-switching-codesaviours-si26-qandeel` created, README added, this notebook pushed as `SI26-Week6-Qandeel.ipynb`
- [ ] `dataset.csv` has 150+ sentences (word-level rows) — ✅ generated above
- [ ] Skimmed `df_final.head(40)` and fixed any obviously wrong auto-labels
- [ ] Hugging Face dataset created, set to Public, dataset card written with source attribution
- [ ] Paste GitHub repo link + Hugging Face dataset link into Classroom
